# 🚀 Lakshya Resume NER Training - Complete Workflow

**DeBERTa-v3-base** model for Named Entity Recognition on resume data

---

## 📊 Dataset Info
- **Training**: 15,433 sentences
- **Test**: 1,930 sentences
- **Labels**: 29 (PERSON_NAME, COMPANY, ROLE, DEGREE, etc.)

## ⏱️ Training Time
- **T4 GPU**: ~60-70 minutes
- **8 epochs**
- **Expected F1**: ~96%

---
## ✅ STEP 1: Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("❌ NO GPU DETECTED!")
    print("")
    print("Please enable GPU:")
    print("1. Runtime > Change runtime type")
    print("2. Hardware accelerator > T4 GPU")
    print("3. Save")
    raise SystemExit("GPU required for training")

---
## 📤 STEP 2: Upload ZIP File

In [ ]:
from google.colab import files
import os

# Clean workspace
!rm -rf /content/Lakshya-Colab-Training /content/*.zip
print("🧹 Cleaned workspace\n")

# Upload ZIP
print("📤 Please upload Lakshya-Colab-Training.zip...")
uploaded = files.upload()

if uploaded:
    zip_name = list(uploaded.keys())[0]
    print(f"\n✅ Uploaded: {zip_name}")
    print(f"   Size: {len(uploaded[zip_name]) / 1024:.1f} KB")
else:
    raise SystemExit("No file uploaded!")

---
## 📦 STEP 3: Extract and Navigate

In [ ]:
import os

# Change to /content first
os.chdir('/content')

# Extract
print("📦 Extracting...")
!unzip -q Lakshya-Colab-Training.zip

# Navigate to ai-service
%cd /content/Lakshya-Colab-Training/ai-service

# Verify structure
print("\n📂 Package structure:")
!ls -lh training/
print("")
!ls -lh training/data/

print("\n✅ Ready to train!")

---
## 🔧 STEP 4: Install Dependencies

In [ ]:
print("📦 Installing dependencies...\n")

!pip install -q transformers==4.44.0 \
                datasets==2.19.0 \
                accelerate==0.33.0 \
                evaluate==0.4.1 \
                seqeval \
                scikit-learn \
                torch==2.4.0

print("\n✅ Dependencies installed")

# Verify installations
import transformers
import datasets
import torch

print(f"\n📚 Versions:")
print(f"   transformers: {transformers.__version__}")
print(f"   datasets: {datasets.__version__}")
print(f"   torch: {torch.__version__}")
print(f"   CUDA: {torch.version.cuda}")

---
## 🏋️ STEP 5: Start Training

**This will take ~60-70 minutes on T4 GPU**

In [ ]:
print("\n🚀 Starting DeBERTa training...")
print("⏱️  Expected time: ~60-70 minutes on T4 GPU")
print("="*60)

!python training/train_colab_standalone.py

---
## 📊 STEP 6: Verify Training Results

In [ ]:
import os
import json

model_dir = "models/resume-ner-deberta"

if os.path.exists(model_dir):
    print("✅ Model training completed!\n")
    
    print("📂 Model files:")
    !ls -lh {model_dir}/
    
    # Load and display label mappings
    label_file = f"{model_dir}/label_mappings.json"
    if os.path.exists(label_file):
        with open(label_file, 'r') as f:
            mappings = json.load(f)
        print(f"\n🏷️  Labels ({len(mappings['labels'])} total):")
        print(f"   {', '.join(mappings['labels'][:10])}...")
    
    # Get model size
    total_size = sum(os.path.getsize(os.path.join(model_dir, f)) 
                     for f in os.listdir(model_dir) 
                     if os.path.isfile(os.path.join(model_dir, f)))
    print(f"\n💾 Total model size: {total_size / 1024 / 1024:.1f} MB")
else:
    print("❌ Model directory not found!")
    print("   Training may have failed. Check the output above.")

---
## 💾 STEP 7: Save to Google Drive (Optional)

In [ ]:
from google.colab import drive

print("💾 Mounting Google Drive...")
drive.mount('/content/drive')

# Create directory in Drive
!mkdir -p "/content/drive/MyDrive/Resume-NER-Models"

# Copy model to Drive
print("\n📤 Copying model to Google Drive...")
!cp -r models/resume-ner-deberta "/content/drive/MyDrive/Resume-NER-Models/"

print("\n✅ Model saved to Google Drive!")
print("   Location: /MyDrive/Resume-NER-Models/resume-ner-deberta")

# Verify
print("\n📂 Files in Drive:")
!ls -lh "/content/drive/MyDrive/Resume-NER-Models/resume-ner-deberta"

---
## 📥 STEP 8: Download Model ZIP

In [ ]:
from google.colab import files

print("📦 Creating ZIP file...")
!cd models && zip -r resume-ner-deberta.zip resume-ner-deberta

# Get ZIP size
import os
zip_path = "models/resume-ner-deberta.zip"
if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f"\n✅ ZIP created: {size_mb:.1f} MB")
    
    print("\n📥 Downloading...")
    files.download(zip_path)
    print("\n✅ Download started! Check your browser downloads.")
else:
    print("❌ ZIP file not created!")

---
## 🧪 STEP 9: Test the Model (Optional)

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

print("🔄 Loading trained model...")
model_path = "models/resume-ner-deberta"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)
model.eval()

# Test sentence
test_text = "John Doe worked as Senior Software Engineer at Google from January 2020 to Present"
print(f"\n📝 Test sentence:\n   {test_text}\n")

# Tokenize and predict
inputs = tokenizer(test_text, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=2)

# Get tokens and labels
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
labels = [model.config.id2label[p.item()] for p in predictions[0]]

# Display results
print("🎯 Predictions:")
print("="*60)
for token, label in zip(tokens, labels):
    if token not in ['[CLS]', '[SEP]', '[PAD]']:
        print(f"   {token:20s} -> {label}")
print("="*60)

print("\n✅ Model is working!")

---
## 📊 Training Summary

### Expected Results:
- **Precision**: ~97%
- **Recall**: ~96%
- **F1 Score**: ~96%

### Model Details:
- **Base Model**: microsoft/deberta-v3-base (184M parameters)
- **Task**: Token Classification (NER)
- **Labels**: 29 entity types
- **Training Data**: 15,433 sentences
- **Test Data**: 1,930 sentences

### Next Steps:
1. ✅ Download the model ZIP
2. Extract it in your local project
3. Replace the old model in `ai-service/models/resume-ner-deberta/`
4. Restart your AI service
5. Test with real resumes!

---
**Training Complete! 🎉**